# RFW frozen-codec 1:1 verification

LFW 또는 SurvFace development split에서 이미 fit되어 SHA가 고정된 PCA/PQ codec을 RFW origin embeddings에 그대로 적용한다. RFW 9-fold에서 threshold를 정하고 held-out fold를 평가하며, 결과는 supplementary 1:1 verification으로만 보고한다.


In [ ]:
from pathlib import Path
import sys

def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "research").is_dir() and (candidate / "configs").is_dir():
            return candidate
    raise FileNotFoundError("C:/ronbun project root could not be located")

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from research.experiments import (
    evaluate_rfw_frozen_codecs,
    frozen_codec_specs_from_completed_run,
    load_rfw_frozen_codec_evaluation,
    rfw_frozen_codec_evaluation_uid,
)


## 실행 설정과 frozen codec lineage

`CODEC_SOURCE_RUN_DIRS`에 사용자가 선택한 완료 LFW/SurvFace run을 명시한다. 노트북은 최신 run을 자동 선택하지 않으며, run의 완료 상태, model UID, frozen codec manifest와 모든 SHA를 검증해 `FrozenCodecSpec`을 만든다. 기존 run에 codec manifest가 없으면 origin-only baseline까지만 실행할 수 있다.


In [ ]:
MODEL_UID = "edgeface-a348c305af33c223b337"
ARTIFACT_STORAGE_MODE = "results_only"
EXECUTE_STAGE = True
WRITE_OUTPUTS = True
REUSE_COMPLETED = True
BOOTSTRAP_SEED = 42
BOOTSTRAP_REPEATS = 2000

# 자동 latest 선택은 금지한다. 같은 MODEL_UID로 새로 완료된 run만 넣는다.
CODEC_SOURCE_RUN_DIRS = ()
# CODEC_SOURCE_RUN_DIRS = (
#     PROJECT_ROOT / "runs/lfw_YYYYMMDD/<completed-edgeface-run>",
#     PROJECT_ROOT / "runs/survface_YYYYMMDD/<completed-edgeface-run>",
# )
SELECTED_CODEC_FAMILIES = ("pca", "pq")
SELECTED_CODEC_PROFILES = None  # 예: ("pca_32", "pq_m128_b8")
ALLOW_ORIGIN_ONLY = True

if ARTIFACT_STORAGE_MODE not in {"results_only", "full"}:
    raise ValueError("ARTIFACT_STORAGE_MODE must be results_only or full")
if WRITE_OUTPUTS and not EXECUTE_STAGE:
    raise ValueError("WRITE_OUTPUTS=True requires EXECUTE_STAGE=True")
ARTIFACT_ROOT = PROJECT_ROOT / ("results" if ARTIFACT_STORAGE_MODE == "results_only" else "runs")
ORIGIN_ARTIFACT_DIR = ARTIFACT_ROOT / "rfw_step7/origin_embeddings" / MODEL_UID
OUTPUT_ROOT = ARTIFACT_ROOT / "rfw_step7/frozen_codec_evaluation" / MODEL_UID


## 평가 실행

PCA는 실제 reduced-coordinate cosine과 reconstruction cosine을 분리한다. PQ는 reconstruction cosine과 symmetric ADC-like negative squared-L2를 분리한다. codec artifact 자체의 실제 file bytes도 전체 저장량에 포함한다.


In [ ]:
codec_specs = []
for source_run_dir in CODEC_SOURCE_RUN_DIRS:
    codec_specs.extend(
        frozen_codec_specs_from_completed_run(
            source_run_dir,
            expected_model_uid=MODEL_UID,
            families=SELECTED_CODEC_FAMILIES,
            profile_names=SELECTED_CODEC_PROFILES,
        )
    )
codec_specs = tuple(codec_specs)
if not codec_specs and not ALLOW_ORIGIN_ONLY:
    raise RuntimeError(
        "No frozen codecs selected. Add explicit completed run paths or "
        "set ALLOW_ORIGIN_ONLY=True."
    )

evaluation_uid = rfw_frozen_codec_evaluation_uid(
    origin_artifact_dir=ORIGIN_ARTIFACT_DIR,
    codec_specs=codec_specs,
    strict_official=True,
    bootstrap_seed=BOOTSTRAP_SEED,
    bootstrap_repeats=BOOTSTRAP_REPEATS,
)
OUTPUT_DIR = OUTPUT_ROOT / evaluation_uid
evaluation = None
if EXECUTE_STAGE and WRITE_OUTPUTS:
    evaluation = evaluate_rfw_frozen_codecs(
        origin_artifact_dir=ORIGIN_ARTIFACT_DIR,
        codec_specs=codec_specs,
        output_dir=OUTPUT_DIR,
        strict_official=True,
        bootstrap_seed=BOOTSTRAP_SEED,
        bootstrap_repeats=BOOTSTRAP_REPEATS,
        reuse_completed=REUSE_COMPLETED,
    )
elif (OUTPUT_DIR / "_SUCCESS").is_file():
    evaluation = load_rfw_frozen_codec_evaluation(OUTPUT_DIR)

{
    "evaluation_uid": evaluation_uid,
    "output_dir": str(OUTPUT_DIR),
    "codec_count": len(codec_specs),
    "status": "completed" if evaluation else "planned",
    "profile_summary": (
        evaluation.profile_summary if evaluation is not None else None
    ),
}


## 해석 경계

RFW 결과에는 verification accuracy, TAR/FAR와 group gap/CI만 사용한다. 현재 evaluator가 EER를 산출하지 않으므로 EER를 결과로 기재하지 않는다. DIR@FPIR, open-set rank 또는 RFW에 적합한 codec이라는 표현은 사용하지 않는다. codec fit이 RFW에서 수행되지 않았음은 manifest의 `fit_on_rfw=false`로 확인한다. `codec_count=0` 결과는 origin-only baseline이며 압축 일반화 결과가 아니다.
